In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/playground-series-s6e5


In [3]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv")

print(train.shape)
print(test.shape)
print(train.head())

(439140, 16)
(188165, 15)
   id Driver Compound                   Race  Year  PitStop  LapNumber  Stint  \
0   0   D109     HARD    Canadian Grand Prix  2022        0         50      2   
1   1   D086     HARD       Dutch Grand Prix  2025        1         27      2   
2   2    ZON     HARD    Austrian Grand Prix  2022        0         59      3   
3   3    SPE   MEDIUM     Pre-Season Testing  2023        0          2      1   
4   4   D019     HARD  Azerbaijan Grand Prix  2022        1         26      3   

   TyreLife  Position  LapTime (s)  LapTime_Delta  Cumulative_Degradation  \
0      39.0         8       78.491         -7.564                  21.019   
1       7.0         4       75.095        -32.617                -223.207   
2      22.0        13       70.945         -7.540                -100.529   
3       2.0         7       94.361         -7.324                  -7.324   
4       6.0         2      107.878          8.965                 -14.139   

   RaceProgress  Positio

In [4]:
print(train.columns.tolist())
print(train.info())
print(train['PitNextLap'].value_counts(normalize=True))

['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439

In [5]:
print(train.head())
print(train.info())
print(train['PitNextLap'].value_counts(normalize=True))

   id Driver Compound                   Race  Year  PitStop  LapNumber  Stint  \
0   0   D109     HARD    Canadian Grand Prix  2022        0         50      2   
1   1   D086     HARD       Dutch Grand Prix  2025        1         27      2   
2   2    ZON     HARD    Austrian Grand Prix  2022        0         59      3   
3   3    SPE   MEDIUM     Pre-Season Testing  2023        0          2      1   
4   4   D019     HARD  Azerbaijan Grand Prix  2022        1         26      3   

   TyreLife  Position  LapTime (s)  LapTime_Delta  Cumulative_Degradation  \
0      39.0         8       78.491         -7.564                  21.019   
1       7.0         4       75.095        -32.617                -223.207   
2      22.0        13       70.945         -7.540                -100.529   
3       2.0         7       94.361         -7.324                  -7.324   
4       6.0         2      107.878          8.965                 -14.139   

   RaceProgress  Position_Change  PitNextLap  
0  

In [6]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

X = train.drop(columns=["PitNextLap"])
y = train["PitNextLap"]

X_test = test.copy()

cat_features = X.select_dtypes(include=["object"]).columns.tolist()

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=500,
        random_seed=42
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    oof[valid_idx] = model.predict_proba(X_valid)[:, 1]
    preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

print("CV AUC:", roc_auc_score(y, oof))

0:	test: 0.9049335	best: 0.9049335 (0)	total: 424ms	remaining: 21m 12s
500:	test: 0.9437747	best: 0.9437747 (500)	total: 2m 42s	remaining: 13m 32s
1000:	test: 0.9471386	best: 0.9471386 (1000)	total: 5m 22s	remaining: 10m 44s
1500:	test: 0.9485282	best: 0.9485282 (1500)	total: 8m 2s	remaining: 8m 2s
2000:	test: 0.9491056	best: 0.9491060 (1998)	total: 10m 42s	remaining: 5m 20s
2500:	test: 0.9495473	best: 0.9495473 (2500)	total: 13m 21s	remaining: 2m 39s
2999:	test: 0.9497991	best: 0.9497999 (2987)	total: 16m	remaining: 0us

bestTest = 0.9497998611
bestIteration = 2987

Shrink model to first 2988 iterations.
0:	test: 0.9024527	best: 0.9024527 (0)	total: 330ms	remaining: 16m 30s
500:	test: 0.9415937	best: 0.9415937 (500)	total: 2m 40s	remaining: 13m 22s
1000:	test: 0.9450782	best: 0.9450782 (1000)	total: 5m 17s	remaining: 10m 33s
1500:	test: 0.9464263	best: 0.9464263 (1500)	total: 7m 55s	remaining: 7m 55s
2000:	test: 0.9470353	best: 0.9470353 (2000)	total: 10m 34s	remaining: 5m 16s
2500:	t

In [7]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": preds
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,PitNextLap
0,439140,0.006523
1,439141,0.004925
2,439142,0.007228
3,439143,0.104904
4,439144,0.869569


In [8]:
print(train.shape)
print(train.columns.tolist())
train.head()

(439140, 16)
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("CV AUC:", roc_auc_score(y, oof))

CV AUC: 0.9485943132284548
